# Dataset Selection
**TMCD 2025/2026 | 2nd Semester**

This notebook documents an analysis of the four available datasets for the project, leading to the selection of the final dataset for the following tasks. the analysis includes statistical summaries, distribution of labels, train and test size ratio and any other relevant characteristics that may influence the decision.

The four datasets are:
- **Tweets EN** 
    - ~50k English tweets labelled `pos`/`neg`
- **Amazon Reviews** 
    - ~50k product reviews labelled `positive`/`negative`
- **IMDB Reviews** 
    - ~44k movie reviews labelled `pos`/`neg`
- **Rotten Tomatoes** 
    - ~8.5k critic review sentences labelled `positive`/`negative`/`neutral`

## 0. Setup

In [1]:
import os

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

## Data Loading

A dict with keys `train` and `test`, each a `pd.DataFrame` with columns `text` and `label`. 
Rotten Tomatoes uses a tab-separated format with no header.

In [5]:
DATA_DIR = os.path.join('..', 'data')

def load_csv(path, text_col, label_col):
    """Load a CSV into a standardised DataFrame."""
    df = pd.read_csv(path)
    return df.rename(columns={text_col: 'text', label_col: 'label'})[['text', 'label']]

def load_tsv_no_header(path):
    """Load a headerless TSV with format: id \t label \t text."""
    rows = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) >= 3:
                rows.append({'text': parts[2], 'label': parts[1]})
    return pd.DataFrame(rows)

datasets = {
    'Tweets': {
        'train': load_csv(os.path.join(DATA_DIR, 'Tweets_EN_sentiment_train.csv'), 'text', 'class'),
        'test':  load_csv(os.path.join(DATA_DIR, 'Tweets_EN_sentiment_test.csv'),  'text', 'class'),
    },
    'Amazon': {
        'train': load_csv(os.path.join(DATA_DIR, 'amazon_reviews_train.csv'), 'review', 'sentiment'),
        'test':  load_csv(os.path.join(DATA_DIR, 'amazon_reviews_test.csv'),  'review', 'sentiment'),
    },
    'IMDB': {
        'train': load_csv(os.path.join(DATA_DIR, 'imdb_reviews_train.csv'), 'text', 'label'),
        'test':  load_csv(os.path.join(DATA_DIR, 'imdb_reviews_test.csv'),  'text', 'label'),
    },
    'Rotten Tomatoes': {
        'train': load_tsv_no_header(os.path.join(DATA_DIR, 'rotten_tomatoes_train.tsv')),
        'test':  load_tsv_no_header(os.path.join(DATA_DIR, 'rotten_tomatoes_test.tsv')),
    },
}

# Add word count column to every split
for ds in datasets.values():
    for split in ds.values():
        split['word_count'] = split['text'].str.split().str.len()

for name, ds in datasets.items():
    print(f"{name}: train={len(ds['train']):,}  test={len(ds['test']):,}")

Tweets: train=39,936  test=9,985
Amazon: train=48,902  test=2,417
IMDB: train=21,754  test=21,996
Rotten Tomatoes: train=6,800  test=1,728


## Statistics

In [ ]:
rows = []
for name, ds in datasets.items():
    for split_name, df in ds.items():
        label_counts = df['label'].value_counts().to_dict()
        total = len(df)
        majority = max(label_counts.values()) / total * 100
        rows.append({
            'Dataset':       name,
            'Split':         split_name,
            'Total':         total,
            'Classes':       len(label_counts),
            'Label counts':  str(label_counts),
            'Majority (%)':  round(majority, 1),
            'Avg words':     round(df['word_count'].mean(), 1),
            'Min words':     int(df['word_count'].min()),
            'Max words':     int(df['word_count'].max()),
        })

summary = pd.DataFrame(rows)
pd.set_option('display.max_colwidth', 60)
summary

,Dataset,Split,Total,Classes,Label counts,Majority (%),Avg words,Min words,Max words
0,Tweets,train,39936,2,"{'pos': 33065, 'neg': 6871}",82.8,13.2,0,34
1,Tweets,test,9985,2,"{'pos': 8312, 'neg': 1673}",83.2,13.1,0,35
2,Amazon,train,48902,2,"{'positive': 37835, 'negative': 11067}",77.4,76.4,7,430
3,Amazon,test,2417,2,"{'positive': 1676, 'negative': 741}",69.3,75.1,6,392
4,IMDB,train,21754,2,"{'neg': 10978, 'pos': 10776}",50.5,178.9,10,463
5,IMDB,test,21996,2,"{'neg': 11050, 'pos': 10946}",50.2,177.6,4,466
6,Rotten Tomatoes,train,6800,3,"{'positive': 2888, 'negative': 2601, 'neutral': 1311}",42.5,19.0,1,52
7,Rotten Tomatoes,test,1728,3,"{'positive': 714, 'negative': 670, 'neutral': 344}",41.3,19.1,2,49


## Train/Test Size Ratio

In [15]:
ratio_data = {
    name: {'Train': len(ds['train']), 'Test': len(ds['test'])}
    for name, ds in datasets.items()
}
ratio_df = pd.DataFrame(ratio_data).T
ratio_df['Test / Train (%)'] = (ratio_df['Test'] / ratio_df['Train'] * 100).round(1)
ratio_df['Total'] = ratio_df['Train'] + ratio_df['Test']
ratio_df

,Train,Test,Test / Train (%),Total
Tweets,39936,9985,25.0,49921
Amazon,48902,2417,4.9,51319
IMDB,21754,21996,101.1,43750
Rotten Tomatoes,6800,1728,25.4,8528


## Dataset Summary

In [16]:
criteria = pd.DataFrame({
    'Dataset':              ['Tweets', 'Amazon', 'IMDB', 'Rotten Tomatoes'],
    'Task type':            ['Binary', 'Binary', 'Binary', '3-class'],
    'Total samples':        ['~50k', '~51k', '~44k', '~8.5k'],
    'Class balance':        ['83/17', '77/23', '50/50', '42/38/19'],
    'Majority baseline (%)':['83.2', '77.4', '50.5', '42.5'],
    'Avg text length':      ['~13 words', '~76 words', '~178 words', '~18 words'],
    'Train/test ratio':     ['~80/20', '~95/5', '~50/50', '~80/20'],
    'Noise / challenges':   ['Informal lang, encoding issues', 'Small test set', 'None significant', '3 classes, very short texts'],
})
criteria.set_index('Dataset', inplace=True)
criteria.style.set_properties(**{'text-align': 'left'}).set_table_styles(
    [{'selector': 'th', 'props': [('text-align', 'left')]}]
)

,Task type,Total samples,Class balance,Majority baseline (%),Avg text length,Train/test ratio,Noise / challenges
Dataset,,,,,,,
Tweets,Binary,~50k,83/17,83.2,~13 words,~80/20,"Informal lang, encoding issues"
Amazon,Binary,~51k,77/23,77.4,~76 words,~95/5,Small test set
IMDB,Binary,~44k,50/50,50.5,~178 words,~50/50,None significant
Rotten Tomatoes,3-class,~8.5k,42/38/19,42.5,~18 words,~80/20,"3 classes, very short texts"
